In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. 데이터셋 준비 (NumPy 데이터를 PyTorch Tensor로 변환)
# 입력 데이터 (9개 샘플, 2개 특성)
X = torch.tensor([
    [1.0, 2.0], [1.5, 1.8], [0.8, 2.5],  # Class 0
    [8.0, 8.0], [7.5, 9.0], [8.5, 7.8],  # Class 1
    [1.0, 8.0], [1.2, 9.0], [0.5, 8.5]   # Class 2
], dtype=torch.float32)

# 타겟 정답 (PyTorch CrossEntropyLoss는 원-핫 인코딩이 아닌 클래스 인덱스(0, 1, 2)를 전달받습니다)
y = torch.tensor([0, 0, 0, 1, 1, 1, 2, 2, 2], dtype=torch.long)


# 2. 신경망 모델 정의 (nn.Module 상속)
class MultiClassNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MultiClassNet, self).__init__()
        # 계층 정의
        self.fc1 = nn.Linear(input_size, hidden_size)  # 입력층 -> 은닉층
        self.sigmoid = nn.Sigmoid()                    # 활성화 함수
        self.fc2 = nn.Linear(hidden_size, output_size) # 은닉층 -> 출력층

    def forward(self, x):
        out = self.fc1(x)
        out = self.sigmoid(out)
        out = self.fc2(out)  # Softmax는 nn.CrossEntropyLoss 내부에서 처리되므로 생략
        return out

# 3. 모델, 손실 함수, 옵티마이저 생성
input_size = 2
hidden_size = 5
output_size = 3
learning_rate = 0.5

# 재현성을 위한 시드 고정
torch.manual_seed(42)

model = MultiClassNet(input_size, hidden_size, output_size)

# 손실 함수: Softmax + Cross-Entropy가 결합된 형태
criterion = nn.CrossEntropyLoss()

# 최적화 알고리즘: 경사하강법(SGD)
optimizer = optim.SGD(model.parameters(), lr=learning_rate)


# 4. 모델 학습 루프 (Training Loop)
print("=== PyTorch 학습 시작 ===")
epochs = 3000

for epoch in range(epochs):
    # ① 순전파 (Forward)
    outputs = model(X)
    loss = criterion(outputs, y)

    # ② 역전파 (Backward) 및 가중치 업데이트
    optimizer.zero_grad()  # 이전 스텝의 기울기(Gradient) 초기화
    loss.backward()        # 자동 미분을 통해 역전파 수행 (autograd)
    optimizer.step()       # 경사하강법으로 가중치 업데이트

    # 출력
    if (epoch + 1) % 500 == 0:
        # 가장 높은 확률 값을 가진 클래스 인덱스 추출
        _, predicted = torch.max(outputs, 1)
        accuracy = (predicted == y).float().mean() * 100
        print(f"Epoch {epoch + 1:4d} | Loss: {loss.item():.4f} | Accuracy: {accuracy.item():.1f}%")


# 5. 테스트 샘플 예측
print("\n=== 테스트 샘플 예측 ===")
model.eval() # 평가 모드 전환
test_sample = torch.tensor([[1.2, 2.1], [8.1, 8.2], [0.9, 8.8]], dtype=torch.float32)

with torch.no_grad(): # 테스트 단계에서는 기울기 계산 불필요
    logits = model(test_sample)
    # 실제 확률 분포를 보고 싶다면 torch.softmax 적용
    probabilities = torch.softmax(logits, dim=1)
    predictions = torch.argmax(probabilities, dim=1)

for i, (prob, pred) in enumerate(zip(probabilities, predictions)):
    prob_list = [round(p, 3) for p in prob.tolist()]
    print(f"샘플 {i+1} 확률 분포: {prob_list} -> 최종 예측 클래스: Class {pred.item()}")

=== PyTorch 학습 시작 ===
Epoch  500 | Loss: 0.0097 | Accuracy: 100.0%
Epoch 1000 | Loss: 0.0043 | Accuracy: 100.0%
Epoch 1500 | Loss: 0.0027 | Accuracy: 100.0%
Epoch 2000 | Loss: 0.0020 | Accuracy: 100.0%
Epoch 2500 | Loss: 0.0016 | Accuracy: 100.0%
Epoch 3000 | Loss: 0.0013 | Accuracy: 100.0%

=== 테스트 샘플 예측 ===
샘플 1 확률 분포: [0.999, 0.001, 0.0] -> 최종 예측 클래스: Class 0
샘플 2 확률 분포: [0.001, 0.999, 0.001] -> 최종 예측 클래스: Class 1
샘플 3 확률 분포: [0.001, 0.001, 0.999] -> 최종 예측 클래스: Class 2


In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

   sepal length (cm)  sepal width (cm)  ...  petal width (cm)  target
0                5.1               3.5  ...               0.2       0
1                4.9               3.0  ...               0.2       0
2                4.7               3.2  ...               0.2       0
3                4.6               3.1  ...               0.2       0
4                5.0               3.6  ...               0.2       0

[5 rows x 5 columns]


In [10]:
# 1. iris 데이터 불러오기
data = load_iris(as_frame=True)['frame']
print(data.info())
print(data.head())

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sepal length (cm)  150 non-null    float64
 1   sepal width (cm)   150 non-null    float64
 2   petal length (cm)  150 non-null    float64
 3   petal width (cm)   150 non-null    float64
 4   target             150 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 6.0 KB
None
   sepal length (cm)  sepal width (cm)  ...  petal width (cm)  target
0                5.1               3.5  ...               0.2       0
1                4.9               3.0  ...               0.2       0
2                4.7               3.2  ...               0.2       0
3                4.6               3.1  ...               0.2       0
4                5.0               3.6  ...               0.2       0

[5 rows x 5 columns]


In [19]:
print(data[['petal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']])
print(len(data))

     petal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
0                  1.4               3.5                1.4               0.2
1                  1.4               3.0                1.4               0.2
2                  1.3               3.2                1.3               0.2
3                  1.5               3.1                1.5               0.2
4                  1.4               3.6                1.4               0.2
..                 ...               ...                ...               ...
145                5.2               3.0                5.2               2.3
146                5.0               2.5                5.0               1.9
147                5.2               3.0                5.2               2.0
148                5.4               3.4                5.4               2.3
149                5.1               3.0                5.1               1.8

[150 rows x 4 columns]
150


In [41]:
# 2. train, test 구분
X_train, X_test, y_train, y_test = train_test_split(
    data[['petal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']],
    data[['target']],
    test_size=0.2,
    random_state=1
)

temp_X = []
for i in range(len(X_train)):
    temp_X.append(list(X_train.iloc[i]))
# print(temp_X)

X = torch.tensor(temp_X)
# print(X)
print(X.shape)

temp_y = list(y_train['target'])
# print(temp_y)

y = torch.tensor(temp_y)
# print(y)
print(y.shape)

torch.Size([120, 4])
torch.Size([120])


In [42]:
# 3. 학습 사전 준비
# 신경망 모델 정의 (nn.Module 상속)
class MultiClassNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MultiClassNet, self).__init__()
        # 계층 정의
        self.fc1 = nn.Linear(input_size, hidden_size)  # 입력층 -> 은닉층
        self.sigmoid = nn.Sigmoid()                    # 활성화 함수
        self.fc2 = nn.Linear(hidden_size, output_size) # 은닉층 -> 출력층

    def forward(self, x):
        out = self.fc1(x)
        out = self.sigmoid(out)
        out = self.fc2(out)  # Softmax는 nn.CrossEntropyLoss 내부에서 처리되므로 생략
        return out

# 모델, 손실 함수, 옵티마이저 생성
input_size = 4
hidden_size = 10
output_size = 120
learning_rate = 0.5

# 재현성을 위한 시드 고정
torch.manual_seed(42)

model = MultiClassNet(input_size, hidden_size, output_size)

# 손실 함수: Softmax + Cross-Entropy가 결합된 형태
criterion = nn.CrossEntropyLoss()

# 최적화 알고리즘: 경사하강법(SGD)
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

In [43]:
# 4. 모델 학습 루프 (Training Loop)
print("=== PyTorch 학습 시작 ===")
epochs = 3000

for epoch in range(epochs):
    # ① 순전파 (Forward)
    outputs = model(X)
    loss = criterion(outputs, y)

    # ② 역전파 (Backward) 및 가중치 업데이트
    optimizer.zero_grad()  # 이전 스텝의 기울기(Gradient) 초기화
    loss.backward()        # 자동 미분을 통해 역전파 수행 (autograd)
    optimizer.step()       # 경사하강법으로 가중치 업데이트

    # 출력
    if (epoch + 1) % 500 == 0:
        # 가장 높은 확률 값을 가진 클래스 인덱스 추출
        _, predicted = torch.max(outputs, 1)
        accuracy = (predicted == y).float().mean() * 100
        print(f"Epoch {epoch + 1:4d} | Loss: {loss.item():.4f} | Accuracy: {accuracy.item():.1f}%")

=== PyTorch 학습 시작 ===
Epoch  500 | Loss: 0.1605 | Accuracy: 93.3%
Epoch 1000 | Loss: 0.1092 | Accuracy: 95.0%
Epoch 1500 | Loss: 0.0956 | Accuracy: 95.0%
Epoch 2000 | Loss: 0.0881 | Accuracy: 95.8%
Epoch 2500 | Loss: 0.0830 | Accuracy: 96.7%
Epoch 3000 | Loss: 0.0795 | Accuracy: 96.7%


In [51]:
# 5. 테스트 샘플 예측
print("\n=== 테스트 샘플 예측 ===")
model.eval() # 평가 모드 전환

# X 테스트 데이터
temp_X = []
for i in range(len(X_test)):
    temp_X.append(list(X_test.iloc[i]))
# print(temp_X)

X_sample_test = torch.tensor(temp_X)
# print(X)
print(X_sample_test.shape)

# y 테스트 데이터
temp_y = list(y_test['target'])
# print(temp_y)

y_sample_test = torch.tensor(temp_y)
# print(y)
print(y_sample_test.shape)

# test_sample = torch.tensor([[1.2, 2.1], [8.1, 8.2], [0.9, 8.8]], dtype=torch.float32)

with torch.no_grad(): # 테스트 단계에서는 기울기 계산 불필요
    logits = model(X_sample_test)

    # 실제 확률 분포를 보고 싶다면 torch.softmax 적용
    probabilities = torch.softmax(logits, dim=1)
    predictions = torch.argmax(probabilities, dim=1)

for i, (prob, pred) in enumerate(zip(probabilities, predictions)):
    prob_list = [round(p, 3) for p in prob.tolist()]
    print(f"샘플 {i+1} -> 최종 예측 클래스: Class {pred.item()} | 실제 값 : {y_sample_test[i]}")


=== 테스트 샘플 예측 ===
torch.Size([30, 4])
torch.Size([30])
샘플 1 -> 최종 예측 클래스: Class 0 | 실제 값 : 0
샘플 2 -> 최종 예측 클래스: Class 1 | 실제 값 : 1
샘플 3 -> 최종 예측 클래스: Class 1 | 실제 값 : 1
샘플 4 -> 최종 예측 클래스: Class 0 | 실제 값 : 0
샘플 5 -> 최종 예측 클래스: Class 2 | 실제 값 : 2
샘플 6 -> 최종 예측 클래스: Class 1 | 실제 값 : 1
샘플 7 -> 최종 예측 클래스: Class 2 | 실제 값 : 2
샘플 8 -> 최종 예측 클래스: Class 0 | 실제 값 : 0
샘플 9 -> 최종 예측 클래스: Class 0 | 실제 값 : 0
샘플 10 -> 최종 예측 클래스: Class 2 | 실제 값 : 2
샘플 11 -> 최종 예측 클래스: Class 1 | 실제 값 : 1
샘플 12 -> 최종 예측 클래스: Class 0 | 실제 값 : 0
샘플 13 -> 최종 예측 클래스: Class 2 | 실제 값 : 2
샘플 14 -> 최종 예측 클래스: Class 1 | 실제 값 : 1
샘플 15 -> 최종 예측 클래스: Class 1 | 실제 값 : 1
샘플 16 -> 최종 예측 클래스: Class 0 | 실제 값 : 0
샘플 17 -> 최종 예측 클래스: Class 1 | 실제 값 : 1
샘플 18 -> 최종 예측 클래스: Class 1 | 실제 값 : 1
샘플 19 -> 최종 예측 클래스: Class 0 | 실제 값 : 0
샘플 20 -> 최종 예측 클래스: Class 0 | 실제 값 : 0
샘플 21 -> 최종 예측 클래스: Class 1 | 실제 값 : 1
샘플 22 -> 최종 예측 클래스: Class 1 | 실제 값 : 1
샘플 23 -> 최종 예측 클래스: Class 2 | 실제 값 : 1
샘플 24 -> 최종 예측 클래스: Class 0 | 실제 값 : 0
샘플 25 -> 최종 예측 클래